# 3DMapping — Free Google Colab Reconstruction

This notebook is the **$0 GPU path** for the 3DMapping project. It takes a drone video, samples frames, runs COLMAP for camera poses, trains a Gaussian Splat with Nerfstudio/Splatfacto, and exports a `.ply` file for the project's browser viewer.

**Important:** Google Colab free GPU availability and runtime limits vary. The runtime is temporary, so download the final `.ply` before the session ends.

In [ ]:
# Check the free runtime GPU. If this prints a CUDA GPU, continue.
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Install the free reconstruction stack.
!apt-get update -qq
!apt-get install -y -qq ffmpeg colmap
!pip -q install --upgrade pip
!pip -q install nerfstudio

print('COLMAP:')
!colmap -h | head -n 8
print('Nerfstudio:')
!ns-train --help | head -n 8

In [ ]:
# Upload a drone video from your Mac.
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No video selected.')
video_name = next(iter(uploaded))
video_path = Path('/content') / video_name
print('Video:', video_path)
!ffprobe -v error -show_entries format=duration -show_entries stream=width,height,r_frame_rate,codec_name -of default=noprint_wrappers=1 "{video_path}"

In [ ]:
# Extract a manageable, sequential image set.
# 3 FPS is a good free-runtime starting point; increase to 4–5 only after a successful test.
import shutil, subprocess, os
from pathlib import Path

root = Path('/content/3dmapping')
images = root / 'images'
if root.exists(): shutil.rmtree(root)
images.mkdir(parents=True)

fps = 3
max_width = 1600
subprocess.run([
    'ffmpeg','-y','-i',str(video_path),
    '-vf',f'fps={fps},scale={max_width}:-2:force_original_aspect_ratio=decrease',
    '-q:v','2', str(images/'frame_%06d.jpg')
], check=True)

count = len(list(images.glob('*.jpg')))
print('Extracted frames:', count)
if count < 20:
    raise RuntimeError('Too few frames were extracted for a useful reconstruction.')


In [ ]:
# COLMAP: feature extraction + sequential matching + incremental SfM.
# Sequential matching is appropriate for ordered video frames.
db = root / 'database.db'
sparse = root / 'sparse'
sparse.mkdir(exist_ok=True)

!colmap feature_extractor --database_path "{db}" --image_path "{images}" --ImageReader.single_camera 1 --FeatureExtraction.use_gpu 1
!colmap sequential_matcher --database_path "{db}" --SequentialMatching.overlap 10 --FeatureMatching.use_gpu 1
!colmap mapper --database_path "{db}" --image_path "{images}" --output_path "{sparse}"

models = sorted([p for p in sparse.iterdir() if p.is_dir() and p.name.isdigit()], key=lambda p:int(p.name))
if not models:
    raise RuntimeError('COLMAP did not produce a sparse model. Try footage with more overlap, more visible texture, and slower camera motion.')
model = models[0]
print('COLMAP model:', model)
print('Model files:', [p.name for p in model.iterdir()])

In [ ]:
# Prepare a Nerfstudio dataset from the COLMAP result.
processed = root / 'ns_data'
if processed.exists(): shutil.rmtree(processed)

# Nerfstudio's process command accepts the original images and COLMAP sparse model.
!ns-process-data images --data "{images}" --output-dir "{processed}" --colmap-model-path "{model}" --skip-image-processing

if not (processed / 'transforms.json').exists():
    raise RuntimeError('Nerfstudio preprocessing did not create transforms.json.')
print('Prepared dataset:', processed)

In [ ]:
# Train Gaussian Splatting.
# Start small for the free Colab runtime. If it succeeds, raise max_num_iterations.
output_dir = root / 'nerfstudio_output'
iterations = 15000

!ns-train splatfacto --data "{processed}" --output-dir "{output_dir}" --max-num-iterations {iterations} --viewer.quit-on-train-completion True

configs = sorted(output_dir.glob('*/**/config.yml'))
if not configs:
    raise RuntimeError('Splatfacto training did not produce a config.yml.')
config = configs[-1]
print('Training config:', config)

In [ ]:
# Export the trained Gaussian Splat as PLY for the 3DMapping viewer.
export_dir = root / 'exported_ply'
export_dir.mkdir(exist_ok=True)
!ns-export gaussian-splat --load-config "{config}" --output-dir "{export_dir}"

ply_files = sorted(export_dir.rglob('*.ply'))
if not ply_files:
    raise RuntimeError('No PLY export was produced.')
ply = ply_files[-1]
print('FINAL PLY:', ply)
print('Size MB:', round(ply.stat().st_size / 1024 / 1024, 2))

In [ ]:
# Download the result to your Mac.
from google.colab import files
files.download(str(ply))

## Result

Put the downloaded `.ply` into the 3DMapping viewer workflow. A successful export proves the real COLMAP → Gaussian Splat pipeline worked; it is not the same thing as the browser's capture-quality score.

If COLMAP fails to register enough images, do not force the splat stage. Improve the capture: maintain overlap, avoid motion blur, keep the subject textured, and orbit/translate around the subject instead of only rotating in place.